In [ ]:
%pip install qiskit qiskit-aer matplotlib

# 🔬 Notebook 3: Superposition

### *Ode to Quantum: Meridian Station Quantum Core Lab*

> Not a coin flip. Not a mystery. A precise, controllable angle.

---


## 🛰️ Mission Briefing


*"You've watched `H` land a qubit on the equator of the Bloch sphere and called it
'superposition.' Time to earn that word."* ECHO opens a new console.

*"A superposition isn't a vague fog of maybe-0-maybe-1. It's an exact, mathematical
mixture with a precise probability attached to each outcome and you're about to
control that mixture with a dial, not a coin toss."*

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Define superposition precisely, in terms of amplitudes and probabilities
- Use a continuously tunable rotation (`Ry`) to place a qubit anywhere between |0⟩ and |1⟩
- Predict measurement probabilities from a rotation angle before running the circuit
- Explain measurement collapse and why it's irreversible


## 🧩 Prerequisites

- Notebook 1 - statevectors and ket notation
- Notebook 2 - gates, measurement, shots, and histograms


## 💡 Concept


Superposition means a qubit's state is a weighted combination of `|0⟩` and `|1⟩` at
the same time:

$$ |ψ\rangle = \alpha|0\rangle + \beta|1\rangle $$

`H` gives you one specific superposition. It's a 50/50 split. But that's just one point
on a continuum. The gate `Ry(θ)` lets you dial in *any* split by choosing an angle
`θ`. At `θ = 0` you get pure `|0⟩`. At `θ = π` you get pure `|1⟩`. Everywhere in
between, you get a genuine superposition, and the probability of each outcome
depends smoothly on `θ`.

## 📊 Visualization


Let's watch the Bloch sphere point sweep from the north pole to the south pole as
`θ` increases from `0` to `π`.


## 🧪 Hands-on Code


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
import numpy as np

def qubit_at_angle(theta):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    return qc

# Halfway between the poles
qc_half = qubit_at_angle(np.pi / 2)
print(qc_half.draw())
plot_bloch_multivector(Statevector(qc_half), title="Ry(π/2)")


`Ry(π/2)` is exactly what `H` does (up to a detail called global phase that doesn't
affect anything you measure): the point sits on the equator, 50/50 between `|0⟩`
and `|1⟩`.

## 📐 Math Lens


The probability of measuring each outcome comes from the **squared magnitude** of
its amplitude:

$$ P(0) = |\alpha|^2 \quad P(1) = |\beta|^2 \quad |\alpha|^2 + |\beta|^2 = 1 $$

For the `Ry(θ)` rotation specifically:

$$ \alpha = \cos\left(\frac{\theta}{2}\right) \quad
\beta = \sin\left(\frac{\theta}{2}\right) $$

So you can predict the measurement probabilities **before running anything**, just
from the angle you chose. Let's check that Qiskit agrees with the formula.

In [ ]:
theta = np.pi / 3   # pick any angle

predicted_p0 = np.cos(theta / 2) ** 2
predicted_p1 = np.sin(theta / 2) ** 2
print(f"Predicted from formula: P(0)={predicted_p0:.4f}  P(1)={predicted_p1:.4f}")

qc = qubit_at_angle(theta)
sv = Statevector(qc)
print("Reported by Qiskit:    ", sv.probabilities_dict())


## 🔁 Experiments


Formulas are nice, but real quantum programs don't get to read the statevector
directly, they measure. Let's confirm the formula holds up under actual
measurement statistics, not just Qiskit's internal bookkeeping.


In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram

def measure_at_angle(theta, shots=2000):
    qc = QuantumCircuit(1, 1)
    qc.ry(theta, 0)
    qc.measure(0, 0)
    backend = AerSimulator()
    tqc = transpile(qc, backend)
    return backend.run(tqc, shots=shots).result().get_counts()

for theta in [0, np.pi/6, np.pi/3, np.pi/2, 2*np.pi/3, np.pi]:
    counts = measure_at_angle(theta)
    p0 = counts.get('0', 0) / sum(counts.values())
    print(f"θ={theta:.3f} rad  ->  measured P(0)≈{p0:.3f}   predicted P(0)={np.cos(theta/2)**2:.3f}")


As `θ` sweeps from `0` to `π`, `P(0)` glides smoothly from `1.0` down to `0.0` - and
the measured values should sit close to the predicted ones. Superposition isn't
mysterious once you have the formula; it's a dial you're now able to read.

## 🚀 Challenge


**Find the angle that gives a 25%/75% split.**

You want `P(1) = 0.75` (and therefore `P(0) = 0.25`). Using the formula
`P(1) = sin²(θ/2)`, solve for `θ`, build the circuit, and confirm with 2000 shots
that your measured probabilities land close to the target.


In [ ]:
# Your turn — solve for theta such that P(1) ≈ 0.75, then build and measure it.


## 🪞 Reflection


Before you measure, a qubit in superposition genuinely holds both possibilities,
weighted by amplitude. The instant you measure, one outcome becomes real and the
other vanishes completely. There's no way to recover the original superposition
afterward, even in principle. This is measurement **collapse**, and it's not a flaw
in our equipment. It's one of the deepest features of quantum mechanics, and it's
the reason quantum algorithms are designed so carefully around *when* and *what*
they measure.

## 📦 Summary

- A superposition is a weighted combination of |0⟩ and |1⟩ with precise amplitudes
- `Ry(θ)` lets you dial in any superposition continuously, from pure |0⟩ to pure |1⟩
- Probabilities come from squared amplitudes: P(0)=cos²(θ/2), P(1)=sin²(θ/2)
- Measurement collapses the superposition irreversibly to one classical outcome


## ➡️ Next Mission


Everything so far has lived on a single qubit. Real quantum programs need many
qubits working together. In **Notebook 4: Multi-Qubit Systems**, you'll combine
qubits, meet the tensor product, and start reading multi-qubit statevectors; the
last piece you need before Notebook 5's entanglement.


---
*End of transmission.*
